In [23]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# 1. TOKENIZER
class Tokenizer:
    """
    The Translator:
    Changes words into numbers (IDs) so the computer can read them,
    and changes numbers back into words so we can read them.
    """
    def __init__(self, text):
        self.words = sorted(set(text.split()))
        self.word2idx = {w: i for i, w in enumerate(self.words)}
        self.idx2word = {i: w for w, i in self.word2idx.items()}
        self.vocab_size = len(self.words)

    def encode(self, text):
        return [self.word2idx[w] for w in text.split() if w in self.word2idx]

    def decode(self, tokens):
        return " ".join([self.idx2word[t] for t in tokens])

# 2. POSITIONAL ENCODING
class PositionalEncoding(layers.Layer):
    """
    The GPS:
    Adds a 'time stamp' or 'position' to each word.
    It tells the model which word comes first, second, and third
    so the sentence order makes sense.
    """
    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe = np.zeros((max_len, d_model))
        position = np.arange(0, max_len)[:, np.newaxis]
        div_term = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)
        self.pe = tf.constant(pe[np.newaxis, ...], dtype=tf.float32)

    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]

# 3. TRANSFORMER BLOCK
class TransformerBlock(layers.Layer):
    """
    The Brain Cell:
    This part looks at a word and figures out which other words
    in the sentence are most important to it (Attention).
    """
    def __init__(self, d_model, num_heads, dff):
        super().__init__()
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.ffn = models.Sequential([
            layers.Dense(dff, activation='relu'),
            layers.Dense(d_model)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, x, training=False):
        attn_output = self.mha(query=x, value=x, key=x, use_causal_mask=True)
        out1 = self.layernorm1(x + attn_output)
        ffn_output = self.ffn(out1)
        return self.layernorm2(out1 + ffn_output)

# 4. MINI GPT MODEL
class MiniGPT(models.Model):
    """
    The Complete Robot:
    This puts the Translator, GPS, and Brain Cells together
    to create a machine that can finish your sentences.
    """
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=2):
        super().__init__()
        self.embedding = layers.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)
        self.blocks = [TransformerBlock(d_model, num_heads, d_model * 4) for _ in range(num_layers)]
        self.fc_out = layers.Dense(vocab_size)

    def call(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        for block in self.blocks:
            x = block(x)
        return self.fc_out(x)

# 5. PREPARE DATA
sentences = [
    "deep learning is powerful",
    "deep learning is fun",
    "machine learning is powerful",
    "machine learning is complex",
    "ai is the future",
    "ai is changing the world",
    "data science is interesting",
    "data science is about patterns",
    "python is a language",
    "python is great for ai",
    "the future is bright",
    "the world is changing fast"
]

tokenizer = Tokenizer(" ".join(sentences))
X, Y = [], []

for s in sentences:
    encoded_seq = tokenizer.encode(s)
    for i in range(1, len(encoded_seq)):
        X.append(encoded_seq[:i])
        Y.append(encoded_seq[1:i+1])

# Padding
X = tf.keras.preprocessing.sequence.pad_sequences(X, padding='post')
Y = tf.keras.preprocessing.sequence.pad_sequences(Y, padding='post')
GLOBAL_MAX_LEN = X.shape[1]

# 6. TRAINING
model = MiniGPT(vocab_size=tokenizer.vocab_size)
model.compile(optimizer=tf.keras.optimizers.Adam(0.005),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

print("Training...")
model.fit(X, Y, epochs=500, verbose=0) # Increased epochs for better accuracy

# 7. GENERATION FUNCTION
def generate_sentence(model, tokenizer, seed_text, next_words=4):
    """
    The Storyteller:
    Takes a starting word and uses the model's brain to guess
    the next few words, one by one, to finish the sentence.
    """
    for _ in range(next_words):
        token_list = tokenizer.encode(seed_text)
        # Use the GLOBAL_MAX_LEN defined earlier
        padded_tokens = tf.keras.preprocessing.sequence.pad_sequences([token_list], maxlen=GLOBAL_MAX_LEN, padding='post')

        predictions = model.predict(padded_tokens, verbose=0)

        # Pull prediction from the last word index
        last_word_idx = len(token_list) - 1
        predicted_id = np.argmax(predictions[0, last_word_idx, :])

        output_word = tokenizer.idx2word[predicted_id]
        seed_text += " " + output_word

        if output_word == "future" or output_word == "powerful": break
    return seed_text

# --- TESTING ---
print("\nPredicting whole sentence:")
print("Input 'deep':", generate_sentence(model, tokenizer, "deep"))
print("Input 'machine':", generate_sentence(model, tokenizer, "machine"))
print("Input 'ai':", generate_sentence(model, tokenizer, "ai"))



Training...

Predicting whole sentence:
Input 'deep': deep learning is powerful
Input 'machine': machine learning is powerful
Input 'ai': ai is changing the world
